In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [3]:
def load_text(path):
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()

    chars = sorted(list(set(text)))
    vocab_size = len(chars)

    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}

    data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

    return data, stoi, itos, vocab_size

In [4]:
def get_batch(data, batch_size, block_size):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x, y

In [5]:
def create_causal_mask(T, device):
    mask = torch.triu(torch.ones(T, T), diagonal=1)
    mask = mask.masked_fill(mask == 1, float('-inf'))
    return mask.to(device)

In [6]:
def self_attention(x, Wq, Wk, Wv, mask):
    B, T, C = x.shape

    Q = x @ Wq
    K = x @ Wk
    V = x @ Wv

    scores = Q @ K.transpose(-2, -1) / math.sqrt(C)
    scores = scores + mask

    weights = F.softmax(scores, dim=-1)

    out = weights @ V
    return out

In [7]:
def multi_head_attention(x, params, num_heads, mask):
    B, T, C = x.shape
    head_dim = C // num_heads

    outputs = []

    for h in range(num_heads):
        start = h * head_dim
        end = (h + 1) * head_dim

        x_h = x[:, :, start:end]

        out_h = self_attention(
            x_h,
            params['Wq'][h],
            params['Wk'][h],
            params['Wv'][h],
            mask
        )
        outputs.append(out_h)

    concat = torch.cat(outputs, dim=-1)
    out = concat @ params['Wo']

    return out

In [8]:
def feedforward(x, W1, W2):
    return (F.relu(x @ W1)) @ W2

In [9]:
def transformer_block(x, params, mask):
    # Attention
    attn_out = multi_head_attention(x, params['attn'], params['num_heads'], mask)
    x = x + attn_out
    x = params['ln1'](x)

    # Feedforward
    ff_out = feedforward(x, params['ff']['W1'], params['ff']['W2'])
    x = x + ff_out
    x = params['ln2'](x)

    return x

In [ ]:
def forward(x, params):
    B, T = x.shape

    tok_emb = params['token_embedding'](x)
    pos_ids = torch.arange(T, device=x.device)
    pos_emb = params['position_embedding'](pos_ids)

    x = tok_emb + pos_emb
    mask = create_causal_mask(T, x.device)

    for block in params['blocks']:
        x = transformer_block(x, block, mask)

    x = params['final_ln'](x)
    logits = x @ params['output_linear']

    return logits

In [11]:
def compute_loss(logits, targets):
    B, T, V = logits.shape
    logits = logits.view(B*T, V)
    targets = targets.view(B*T)
    return F.cross_entropy(logits, targets)

In [ ]:
def train(data, params, steps, batch_size, block_size, optimizer):

    for step in range(steps):
        x, y = get_batch(data, batch_size, block_size)
        logits = forward(x, params)
        loss = compute_loss(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 100 == 0:
            print("step:", step, "loss:", loss.item())

In [13]:
def generate(start_tokens, params, max_new_tokens, block_size):
    x = start_tokens

    for _ in range(max_new_tokens):
        x_cond = x[:, -block_size:]

        logits = forward(x_cond, params)
        logits = logits[:, -1, :]

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        x = torch.cat((x, next_token), dim=1)

    return x

In [ ]:
'''config'''
device = 'cpu'

batch_size = 32
block_size = 128
embed_dim = 128
num_heads = 4
num_layers = 3
ff_dim = 4 * embed_dim
steps = 6000
lr = 3e-4

# ====== LOAD DATA ======
data, stoi, itos, vocab_size = load_text('train.txt')

# ====== INIT PARAMS ======
def rand_param(*shape, scale):
    return nn.Parameter(torch.randn(*shape) * scale)

def init_params():
    params = {}

    params['token_embedding'] = nn.Embedding(vocab_size, embed_dim)
    params['position_embedding'] = nn.Embedding(block_size, embed_dim)

    blocks = []
    for _ in range(num_layers):
        block = {}

        attn = {}
        head_dim = embed_dim // num_heads

        attn['Wq'] = [rand_param(head_dim, head_dim, scale=1.0 / math.sqrt(embed_dim)) for _ in range(num_heads)]
        attn['Wk'] = [rand_param(head_dim, head_dim, scale=1.0 / math.sqrt(embed_dim)) for _ in range(num_heads)]
        attn['Wv'] = [rand_param(head_dim, head_dim, scale=1.0 / math.sqrt(embed_dim)) for _ in range(num_heads)]
        attn['Wo'] = rand_param(embed_dim, embed_dim, scale=1.0 / math.sqrt(embed_dim))

        block['attn'] = attn
        block['num_heads'] = num_heads

        block['ff'] = {
            'W1': rand_param(embed_dim, ff_dim, scale=1.0 / math.sqrt(embed_dim)),
            'W2': rand_param(ff_dim, embed_dim, scale=1.0 / math.sqrt(ff_dim))
        }

        block['ln1'] = nn.LayerNorm(embed_dim)
        block['ln2'] = nn.LayerNorm(embed_dim)

        blocks.append(block)

    params['blocks'] = blocks
    params['final_ln'] = nn.LayerNorm(embed_dim)
    params['output_linear'] = rand_param(embed_dim, vocab_size, scale=1.0 / math.sqrt(embed_dim))

    return params

params = init_params()

# ====== PARAM COLLECTION ======
def get_all_params(params):
    p = []

    p += list(params['token_embedding'].parameters())
    p += list(params['position_embedding'].parameters())

    for block in params['blocks']:
        for h in range(block['num_heads']):
            p.append(block['attn']['Wq'][h])
            p.append(block['attn']['Wk'][h])
            p.append(block['attn']['Wv'][h])

        p.append(block['attn']['Wo'])
        p.append(block['ff']['W1'])
        p.append(block['ff']['W2'])

        p += list(block['ln1'].parameters())
        p += list(block['ln2'].parameters())

    p += list(params['final_ln'].parameters())
    p.append(params['output_linear'])

    return p

optimizer = torch.optim.Adam(get_all_params(params), lr=lr)

# call the training loop
train(data, params, steps, batch_size, block_size, optimizer)

# optional: save.. safe if if you want to continue training in later cells
torch.save(params, 'model.pth')

step: 0 loss: 4.543618202209473
step: 100 loss: 2.6335411071777344
step: 200 loss: 2.5409083366394043
step: 300 loss: 2.4817495346069336
step: 400 loss: 2.481520175933838
step: 500 loss: 2.4435112476348877
step: 600 loss: 2.430108070373535
step: 700 loss: 2.3781914710998535
step: 800 loss: 2.3765013217926025
step: 900 loss: 2.3462040424346924
step: 1000 loss: 2.3003041744232178
step: 1100 loss: 2.2955548763275146
step: 1200 loss: 2.2253661155700684
step: 1300 loss: 2.206078052520752
step: 1400 loss: 2.2280681133270264
step: 1500 loss: 2.2011098861694336
step: 1600 loss: 2.127758264541626
step: 1700 loss: 2.0651209354400635
step: 1800 loss: 2.1255064010620117
step: 1900 loss: 2.0177481174468994
step: 2000 loss: 2.0333454608917236
step: 2100 loss: 1.9707419872283936
step: 2200 loss: 1.9496777057647705
step: 2300 loss: 1.970168948173523
step: 2400 loss: 1.9348868131637573
step: 2500 loss: 1.9178482294082642
step: 2600 loss: 1.8961408138275146
step: 2700 loss: 1.8898285627365112
step: 2800

In [ ]:
import os

# loader
input_path = 'input.txt'
if not os.path.exists(input_path):
    input_path = 'train.txt'

data, stoi, itos, vocab_size = load_text(input_path)

if not os.path.exists('model.pth'):
    raise FileNotFoundError("model.pth not found. Run the training cell first.")

# Safe here because the checkpoint was created locally in this notebook.
params = torch.load('model.pth', map_location='cpu', weights_only=False)

# ====== GENERATE ======
start_char = 'H' if 'H' in stoi else next(iter(stoi))
start = torch.tensor([[stoi[start_char]]], dtype=torch.long)

generated = generate(start, params, max_new_tokens=300, block_size=128)

# decode
out = ''.join([itos[int(i)] for i in generated[0]])
print(out)

Hought's thou they lord?
Than more not my hands, and And Botheerdieng, rests,
And have insonere that the passes; whop me:
Stward-shall me wiss nemes the am less,
It when but of I did not bely to debess,
When palan!' you comen.

BENVOLIO:
Our have purster state, to rewive this I have
we mise the sme, 


In [ ]:
import os

# extended training
extra_steps = 2000

if 'params' not in globals():
    if not os.path.exists('model.pth'):
        raise FileNotFoundError("model.pth not found. Run the training cell first.")
    # Safe here because the checkpoint was created locally in this notebook.
    params = torch.load('model.pth', map_location='cpu', weights_only=False)

if 'data' not in globals():
    data, stoi, itos, vocab_size = load_text('train.txt')

optimizer = torch.optim.Adam(get_all_params(params), lr=lr)
train(data, params, extra_steps, batch_size, block_size, optimizer)

torch.save(params, 'model.pth')

step: 0 loss: 1.6137293577194214
step: 100 loss: 1.576945424079895
step: 200 loss: 1.609645962715149
step: 300 loss: 1.6045870780944824
step: 400 loss: 1.6738765239715576
step: 500 loss: 1.5702780485153198
step: 600 loss: 1.5767990350723267
step: 700 loss: 1.663231372833252
step: 800 loss: 1.6313871145248413
step: 900 loss: 1.6010793447494507
step: 1000 loss: 1.6215351819992065
step: 1100 loss: 1.5742920637130737
step: 1200 loss: 1.5899455547332764
step: 1300 loss: 1.5883196592330933
step: 1400 loss: 1.600003719329834
step: 1500 loss: 1.550188422203064
step: 1600 loss: 1.5607844591140747
step: 1700 loss: 1.6710400581359863
step: 1800 loss: 1.5538110733032227
step: 1900 loss: 1.6536808013916016


In [ ]:
import os

#  CUSTOM PROMPT .. it will bw a pop up to enter it  of sorts.. you can also hard code it here if you want
if 'params' not in globals():
    if not os.path.exists('model.pth'):
        raise FileNotFoundError("model.pth not found. Run the training cell first.")
    # Safe here because the checkpoint was created locally in this notebook.
    params = torch.load('model.pth', map_location='cpu', weights_only=False)

prompt = input("Enter a prompt: ").strip()
if not prompt:
    prompt = "H"

if any(ch not in stoi for ch in prompt):
    if " " in stoi:
        prompt = "".join([ch if ch in stoi else " " for ch in prompt])
    else:
        prompt = "".join([ch for ch in prompt if ch in stoi])

if not prompt:
    raise ValueError("Prompt has no valid characters for the current vocabulary.")

start = torch.tensor([[stoi[ch] for ch in prompt]], dtype=torch.long)
generated = generate(start, params, max_new_tokens=300, block_size=block_size)

out = "".join([itos[int(i)] for i in generated[0]])
print(out)

fellation,
In the same yeing a hand bastardior a shall dlen
Hath refllow thy stick your supptirto-part.

VORMIANDANDIUS:
The could made Pay minamstly
down trief and thereff you, drish apph, I say
At notence the servenn
with fight death of accure his the since,
Have drums thefores, back sosstern of to the us
